# Eksperyment 2: 
## Odszumianie i rekonstrukcja metodą SDEdit.

Celem poniższego eksperymentu jest ocena zdolności trzech różnych architektur sieci neuronowych (Multi-Layer Perceptron, Conv1D oraz U-Net 1D) do dokładnego odszumiania (denoisingu) i rekonstrukcji znormalizowanych funkcji matematycznych, wykorzystując w tym celu modele dyfuzyjne (DDPM) zintegrowane z techniką SDEdit. 

W tej fazie badana jest **odporność modeli na zakłócenia (robustness)** oraz ich zdolność do odzyskania informacji z sygnału celowo zdegradowanego gęstym szumem gaussowskim o zadanej amplitudzie. Eksperyment ten weryfikuje praktyczną użyteczność poszczególnych architektur w warunkach imitujących rzeczywiste, zaszumione pomiary, opierając się na znajomości rozkładu danych wyuczonej przez modele w poprzednim etapie (patrz notebook 1. eksperyment_generacja...).

In [1]:
import numpy as np
import itertools
import time
import pandas as pd
import os
import sys
import copy
import json
import random
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import torch
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter



from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import Image, display, HTML
import pickle

sys.path.append(os.path.abspath('..'))
from utils.samplers import MathFunctions
from utils.metrics import calculate_metrics, generate_comparison_tables, aggregate_experiment_metrics
import utils.visualisations as vis
from models.ddpm1d import DDPM1D, SinusoidalPositionEmbeddings, get_beta_schedule
from models.mlp import DenoiseNet1D_MLP
from models.conv1d import DenoiseNet1D_Conv
from models.unet import DenoiseNet1D_UNet

import warnings
warnings.filterwarnings('ignore') 
%matplotlib inline

### 1. Definicja harmonogramu wariancji szumu (beta schedules)

W modelach dyfuzyjnych typu DDPM, w procesie wprzód (forward process) dodaje się stopniowo sszum gaussowski do oryginalnych danych. Wariancja $\beta_t$ jest parametrem kontrolującym ilość wstrzykiwanego szumu w każdym dyskretnym kroku czasowym $t$. Poniższa komórka implementuje dwa klasyczne harmonogramy przyrostu szumu:
- Linear - harmonogram liniowy, który oznacza stały, jednolity przyrost. Przy dłuższych procesach dyfuzyjnych zbyt gwałtownie degraduje informacje niesione przez sygnał (patrz notebook: 0. wizualizacje_wplywu...) we wczesnych fazach.
- Cosine - harmonogram cosinusowy, który został zaprojektowany z myślą o poprawie stabilności i optymalizacji wyników. Wykorzystuje funkcję trygonometryczną do regulowania wariancji $\alpha_t$, determinuącej zawartość prawdziwego sygnału w zaszumionej próbce. Dzięki czemu informacje zanikają bardziej łagodnie i nieliniowo, przez co model na etapie odszumiania skuteczniej i dokładniej uczy się odtwarzać sygnał 1D.

In [2]:
# import matplotlib.pyplot as plt

# # Testujemy Twoją klasę
# model_emb = SinusoidalPositionEmbeddings(dim=128)
# test_t = torch.arange(100)
# embeddings = model_emb(test_t) # Generujemy dla 100 kroków

# plt.figure(figsize=(10, 6))
# plt.imshow(embeddings.detach().numpy(), aspect='auto', cmap='RdBu')
# plt.colorbar(label="Wartość PE")
# #plt.title("Wizualizacja Embeddingów Czasu")
# plt.xlabel("Wymiar osadzenia (d)")
# plt.ylabel("Krok czasu (t)")
# plt.savefig("../images/embeddingi.png")

### 2. Eksperyment SDEdit

W poniższej komórce znajduje się główny kod eksperymentu (klasa `ExperimentRunner`), którego działanie opiera się na trzech krokach:

1. Celowe zepsucie sygnału - modelowanie zaszumienia.
Na początku bierzemy idealny, czysty sygnał matematyczny i celowo go degradujemy. Dodajemy do niego tak zwany biały szum o określonej sile (`noise_level = 0.4`). 

2. Trening modelu.
W tym kroku sieć neuronowa uczy się, jak naprawiać zepsute dane (tzw. odwrócony łańcuch Markowa). Sieć wielokrotnie przetwarza dane, a specjalny algorytm (optymalizator *Adam*) koryguje jej błędy, starając się zminimalizować funkcję straty. Celem jest, aby model nauczył się krok po kroku odróżniać sztucznie nałożony szum od użytecznych informacji.

3. Właściwe odszumianie metod SDEdit.
To najważniejszy i najbardziej innowacyjny etap eksperymentu. Standardowo modele dyfuzyjne (np. generujące obrazy) zaczynają proces od całkowitego chaosu (czystego szumu) i krok po kroku "wyciągają" z niego dane. My stosujemy podejście:
* Zamiast kazać modelowi zgadywać kształt od zera, dajemy mu podpowiedź. Jako stan startowy podajemy nasz zaszumiony sygnał z pierwszego kroku. 
* Nakazujemy modelowi zacząć proces naprawy zaledwie od **30% drogi** całkowitego procesu odszumiania (`t_start_sdedit = int(0.3 * n_T)`).
Dzięki czemu model wykonuje tylko korektę. Zdąży usunąć drobne, losowe zakłócenia (szum o wysokiej częstotliwości), ale działa na tyle krótko i łagodnie, że nie niszczy głównego, naturalnego kształtu (struktury bazowej) naszego początkowego sygnału. Model traktuje więc zaszumione dane jak szkic, który wystarczy tylko precyzyjnie "wygładzić".

In [3]:
class ExperimentRunner:
    def __init__(self, num_runs=3, device=None, seed=42):
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.seed = seed
        self.num_runs = num_runs
        self.set_seed(self.seed)
        self.math_funcs = MathFunctions(num_points=128)
        
        self.checkpoints_dir = 'experiments/checkpoints2'
        os.makedirs(self.checkpoints_dir, exist_ok=True)
        self.amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    def set_seed(self, seed):
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

    def _ensure_tensor(self, data):
            if isinstance(data, np.ndarray):
                data = torch.from_numpy(data).float()
            if not torch.is_tensor(data):
                data = torch.tensor(data).float()
            
            if data.dim() == 3:
                data = data.squeeze(1)
                
            if data.dim() == 4:
                data = data.squeeze(1).squeeze(-1) 
                
            if data.dim() > 2:
                data = data.view(data.size(0), -1)
                
            return data

    def compute_reconstruction_metrics(self, ddpm, sampler_name, t_start_ratio, skip_steps, n_samples=50):
        ddpm.model.eval()
        with torch.no_grad():
            _, y_true = self.math_funcs.get_dataset(sampler_name, num_samples=n_samples, mode='test')
            y_true_tensor = self._ensure_tensor(y_true).to(self.device)
    
            if y_true_tensor.dim() == 2:
                y_true_tensor = y_true_tensor.unsqueeze(1)  # [50, 128] -> [50, 1, 128]
    
            # t_tensor MUSI być przed a_bar
            t_eval = int(ddpm.n_T * t_start_ratio)
            if t_eval == 0:
                t_eval = 1
            t_tensor = torch.full((n_samples,), t_eval, device=self.device, dtype=torch.long)
    
            a_bar = ddpm.alphas_bar[t_tensor].view(-1, 1, 1)  # [50, 1, 1]
            noise = torch.randn_like(y_true_tensor)            # [50, 1, 128]
            x_noisy = torch.sqrt(a_bar) * y_true_tensor + torch.sqrt(1 - a_bar) * noise  # [50, 1, 128]
    
            start_t = time.time()
            y_gen_tensor = ddpm.ddim_denoise_signal(x_noisy, t_start=t_eval, skip_steps=skip_steps)
            exec_t = (time.time() - start_t) / n_samples
    
            y_true_np = y_true_tensor.cpu().numpy().reshape(n_samples, -1)  # [50, 128]
            y_gen_np = y_gen_tensor.cpu().numpy().reshape(n_samples, -1)    # [50, 128]
    
            all_metrics = []
            for i in range(n_samples):
                m = calculate_metrics(y_true_np[i], y_gen_np[i], exec_time=exec_t)
                all_metrics.append(m)
    
            avg_metrics = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0].keys()}
    
        return avg_metrics
    def optimize_architecture_for_function(self, func_name, model_info, arch_name, t_list, schedules, 
                                           t_start_ratios, lr_list, batch_sizes, skip_steps_list,
                                           max_epochs, eval_every, patience, previous_results=None):
        
        raw_data_train = self.math_funcs.get_dataset(func_name, num_samples=2500, mode='train')
        y_train_raw = raw_data_train[1] 
        raw_data_val = self.math_funcs.get_dataset(func_name, num_samples=500, mode='val')
        y_val_raw = raw_data_val[1]

        train_tensor = self._ensure_tensor(y_train_raw)
        val_tensor = self._ensure_tensor(y_val_raw)

        results = previous_results if previous_results else {
            'config_name': arch_name,
            'function': func_name,
            'trials': [],
            'best_metrics': {'reconstruction_mse': float('inf')}
        }

        combinations = list(itertools.product(t_list, schedules, lr_list, batch_sizes))
        cache_filename = f"experiments/cache/results_cache_{arch_name}_{func_name}.pkl"
        os.makedirs('experiments/cache', exist_ok=True)

        for config in tqdm(combinations, desc=f"Trial Progress ({arch_name})", leave=False):
            n_T, schedule, lr, bs = config
            
            already_done = any(
                t['params'] == {'T': n_T, 'schedule': schedule, 'lr': lr, 'batch_size': bs} 
                for t in results['trials']
            )
            if already_done:
                continue

            config_results = {
                'params': {'T': n_T, 'schedule': schedule, 'lr': lr, 'batch_size': bs},
                'runs': []
            }

            for run_id in range(self.num_runs):
                run_seed = self.seed + run_id 
                self.set_seed(run_seed)
            
                _, y_train_raw = self.math_funcs.get_dataset(func_name, num_samples=2500, mode='train', seed=run_seed)
                _, y_val_raw = self.math_funcs.get_dataset(func_name, num_samples=500, mode='val', seed=run_seed)
                
                t_tensor = self._ensure_tensor(y_train_raw)
                v_tensor = self._ensure_tensor(y_val_raw)

                run_data = self._train_worker(
                    config, run_id, model_info, arch_name, func_name, 
                    t_tensor, v_tensor, max_epochs, eval_every, patience
                )
                
                inference_results = []
                best_mse_in_this_run = float('inf')
                best_metrics_in_this_run = {} # <--- POPRAWKA
                
                for ratio in t_start_ratios:
                    for skip in skip_steps_list:
                        metrics_dict = self.compute_reconstruction_metrics(
                            run_data['final_ddpm'], 
                            func_name, 
                            t_start_ratio=ratio, 
                            skip_steps=skip
                        )
                        metrics_dict['t_start_ratio'] = ratio
                        metrics_dict['skip_steps'] = skip
                        inference_results.append(metrics_dict)
                        
                        if metrics_dict['MSE'] < best_mse_in_this_run:
                            best_mse_in_this_run = metrics_dict['MSE']
                            best_metrics_in_this_run = metrics_dict.copy() # <--- POPRAWKA
                
                run_data['inference_grid'] = inference_results 
                run_data['best_reconstruction_mse'] = best_mse_in_this_run
                run_data['all_metrics'] = best_metrics_in_this_run # <--- POPRAWKA
                
                del run_data['final_ddpm'] 
                config_results['runs'].append(run_data)

            best_run_mses = [r['best_reconstruction_mse'] for r in config_results['runs']]
            config_results['stats'] = {
                'mu_rec_mse': np.mean(best_run_mses),
                'std_rec_mse': np.std(best_run_mses)
            }
            results['trials'].append(config_results)

            # --- POPRAWIONY ZAPIS BEST METRICS ---
            if config_results['stats']['mu_rec_mse'] < results['best_metrics'].get('reconstruction_mse', float('inf')):
                avg_best_metrics = {}
                keys_to_average = config_results['runs'][0]['all_metrics'].keys()
                
                for k in keys_to_average:
                    avg_best_metrics[k] = np.mean([r['all_metrics'][k] for r in config_results['runs']])

                avg_best_metrics['params'] = config_results['params']
                avg_best_metrics['reconstruction_mse'] = config_results['stats']['mu_rec_mse']
                avg_best_metrics['std'] = config_results['stats']['std_rec_mse']

                results['best_metrics'] = avg_best_metrics

            with open(cache_filename, 'wb') as f:
                pickle.dump(results, f)

        return results
        
    def _train_worker(self, config, run_id, model_info, arch_name, func_name, train_tensor, val_tensor, 
                      max_epochs, eval_every, patience):
        n_T, schedule, lr, bs = config
        capacity = model_info['capacity']
        model_class = model_info['class']
    
        try:
            net = model_class(data_dim=128, base_channels=capacity).to(self.device)
        except TypeError:
            net = model_class(data_dim=128, hidden_dim=capacity).to(self.device)

        if self.device.type == 'cuda':
            try:
                net = torch.compile(net, mode='reduce-overhead')
            except Exception:
                pass  

        train_tensor = train_tensor.to(self.device)
        val_tensor = val_tensor.to(self.device)
        
        # Upewnij się że mamy [B, 1, L]
        if train_tensor.dim() == 2:
            train_tensor = train_tensor.unsqueeze(1)
        if val_tensor.dim() == 2:
            val_tensor = val_tensor.unsqueeze(1)
                
        dataloader = DataLoader(TensorDataset(train_tensor), batch_size=bs, shuffle=True, 
                                num_workers=0, pin_memory=False)
        val_dataloader = DataLoader(TensorDataset(val_tensor), batch_size=bs, shuffle=False,
                                    num_workers=0, pin_memory=False)
        
        betas = get_beta_schedule(schedule, 1e-4, 0.02, n_T)
        ddpm = DDPM1D(net, betas, n_T, self.device)
        optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
        scaler = torch.cuda.amp.GradScaler(enabled=(self.device.type == 'cuda'))

        history = {'train_loss': [], 'val_loss': []}
        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0
        current_val_loss = None

        effective_eval_every = eval_every  
        
        epoch_bar = tqdm(range(1, max_epochs + 1), desc=f"Epoki ({arch_name})", leave=False)
        
        for epoch in epoch_bar:
            net.train()
            e_loss = 0
            for batch in dataloader:
                x = batch[0]
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=self.device.type, dtype=self.amp_dtype):
                    loss = ddpm.compute_loss(x)
                scaler.scale(loss).backward()
                
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                e_loss += loss.item()
            
            avg_train_loss = e_loss / len(dataloader)
            history['train_loss'].append(avg_train_loss)
            scheduler.step()

            if epoch % effective_eval_every == 0:
                net.eval()
                v_loss = 0
                with torch.no_grad():
                    with torch.autocast(device_type=self.device.type, dtype=self.amp_dtype):
                        for v_batch in val_dataloader:
                            vx = v_batch[0]
                            v_loss += ddpm.compute_loss(vx).item() * vx.size(0)
                            
                v_loss /= len(val_tensor)
                history['val_loss'].append(v_loss)
                current_val_loss = v_loss

                if v_loss < best_val_loss:
                    best_val_loss = v_loss
                    best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if patience_counter >= (patience // effective_eval_every):
                    epoch_bar.write(f"Early stopping na epoce {epoch} (Val Loss: {v_loss:.4f})")
                    break

            if current_val_loss is not None:
                epoch_bar.set_postfix({
                    'Train': f"{avg_train_loss:.4f}", 
                    'Val': f"{current_val_loss:.4f}"
                })
                
        epoch_bar.close()
        
        if best_state is not None:
            net.load_state_dict(best_state)
            
        save_path = os.path.join(self.checkpoints_dir, f"{arch_name}_{func_name}_T{n_T}_{schedule}_run{run_id}_best_model.pth")
        torch.save({'model_state_dict': best_state}, save_path)
        
        return {
            'history': history,
            'best_val_loss': best_val_loss,
            'final_ddpm': ddpm, 
            'best_state': best_state
        }


In [4]:

architectures_config = {
    # GRUPA 1: MODELE OPTYMALNE (Najniższy błąd MSE i najlepsi reprezentanci)
    
    'UNet_C128_5e-4':   {'class': DenoiseNet1D_UNet, 'capacity': 128, 'lr': 5e-4},  # Najlepszy wynik dla Sine, Chirp
    'UNet_C256_5e-4':   {'class': DenoiseNet1D_UNet, 'capacity': 256, 'lr': 5e-4},  # Najlepszy dla Hard
    
    'Conv1D_C128_1e-3': {'class': DenoiseNet1D_Conv, 'capacity': 128, 'lr': 1e-3},  # Najlepszy w klasie Conv1D dla Chirp i Hard
    'Conv1D_C128_5e-4': {'class': DenoiseNet1D_Conv, 'capacity': 128, 'lr': 5e-4},  # Najlepszy w klasie Conv1D dla Sine
    'MLP_C256_1e-3':    {'class': DenoiseNet1D_MLP,  'capacity': 256, 'lr': 1e-3},  # Najlepszy w klasie MLP dla Chirp i Hard
    'MLP_C256_5e-4':    {'class': DenoiseNet1D_MLP,  'capacity': 256, 'lr': 5e-4},  # Najlepszy w klasie MLP dla Sine

    # GRUPA 2: MODELE ANOMALNE (Wąskie gardło informacyjne i przeuczenie)
    
    # --- Modele niedouczone (Underfitting - brak parametrów do odwzorowania fizyki) ---
    'UNet_C64_1e-4':    {'class': DenoiseNet1D_UNet, 'capacity': 64,  'lr': 1e-4},  # Underfitting dla sygnału Chirp
    'Conv1D_C32_5e-4':  {'class': DenoiseNet1D_Conv, 'capacity': 32,  'lr': 5e-4},  # Underfitting dla sygnału Sine
    'MLP_C32_1e-4':    {'class': DenoiseNet1D_MLP,  'capacity': 32, 'lr': 1e-4},  # Najgorsza 
    
    # --- Modele skrajnie przeuczone (Overfitting - zapamiętywanie szumu) ---
    'MLP_C256_1e-4':    {'class': DenoiseNet1D_MLP,  'capacity': 256, 'lr': 1e-4},  # Przeuczenie

}
# 2. TEST FUNCTIONS (11 deterministycznych funkcji z rozdziału metodologii)
test_functions = [
    'square_wave',         # 1. Fala prostokątna (ostre krawędzie)
    'damped_oscillator',   # 2. Oscylator tłumiony (zanik wykładniczy + oscylacja)
    'mixed_freq',          # 3. Sygnał wieloczęstotliwościowy (trend + wysokie częstotliwości)
    # 'chirp',               # 4. Funkcja typu Chirp (zagęszczanie struktur lokalnych)
    # 'sinc',                # 5. Funkcja Sinc (wygasające oscylacje falowe)
    # 'step',                # 6. Krok Heaviside'a (izolowana nieciągłość skokowa)
    # 'abs',                 # 7. Moduł (Wartość bezwzględna - punkt nieróżniczkowalny)
    # 'log10',               # 8. Funkcja logarytmiczna (asymptota pionowa)
    # 'log2',                # 9. Funkcja logarytmiczna o podstawie 2
    # '1_over_x',            # 10. Funkcja odwrotna (rozłączna dziedzina i dwie gałęzie)
    # 'exp'                  # 11. Funkcja eksponencjalna (gwałtowny wzrost monotonny)
]


# 3. PROCESS HYPERPARAMETERS (Konfiguracja silnika dyfuzyjnego SDEdit / DDIM)
HYPERPARAMS = {
    't_steps': [80, 100, 120],                 # Liczba kroków procesu dyfuzji
    'schedules': ['linear', 'cosine'],         # Plany redukcji wariancji (harmonogramy szumu)
    'batch_sizes': [128],                      # Zgodnie z konfiguracją testową Eksperymentu II
    't_start_ratios': [0.2, 0.35, 0.5, 0.7],   # Głębokość zaszumienia startowego dla SDEdit
    'skip_steps': [1, 2, 5]                    # Krok pomijania dla deterministycznego DDIM
}

# 4. TRAINING WORKER PARAMETERS (Wczesne zatrzymanie i parametry optymalizacji)

TRAIN_PARAMS = {
    'max_epochs': 5000,   # Zwiększono limit z uwagi na głębokie badanie modeli granicznych
    'eval_every': 50,     # Częstotliwość ewaluacji na zbiorze walidacyjnym
    'patience': 150       # Kryterium cierpliwości dla procedury Early Stopping
}

N_TRIALS = (len(HYPERPARAMS['t_steps']) * len(HYPERPARAMS['schedules']) * 1 * # To nasze pojedyncze LR z konfiguracji
            len(HYPERPARAMS['batch_sizes']))

CACHE_DIR = 'experiments/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

runner = ExperimentRunner(num_runs=3, seed=42)
master_results = {}

print(f"\n{'='*80}\n| {'ROZPOCZYNAM EKSPERYMENT II (SDEDIT)':^76} |\n{'='*80}")

for func in test_functions:
    master_results[func] = {}
    print(f"\n>>> ANALIZA FUNKCJI: {func.upper()}")
    
    for config_name, info in architectures_config.items():
        cache_path = os.path.join(CACHE_DIR, f"results_cache_{config_name}_{func}.pkl")
        
        # 1. PRÓBA WCZYTANIA POSTĘPU (Mikro-Checkpointing)
        previous_results = None
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    previous_results = pickle.load(f)
                # Sprawdzamy, ile prób już mamy
                done_count = len(previous_results.get('trials', []))
                if done_count >= N_TRIALS:
                    print(f"  [DONE]    {config_name:20}: Wszystkie próby ({done_count}) gotowe.")
                    continue
                else:
                    print(f"  [RESUME]  {config_name:20}: Wznawianie od próby {done_count+1}/{N_TRIALS}")
            except Exception as e:
                print(f"  [WARN]    Błąd wczytywania cache dla {config_name}: {e}. Zaczynam od nowa.")
                previous_results = None

        # 2. WYWOŁANIE OPTYMALIZACJI (z przekazaniem postępu)
        results = runner.optimize_architecture_for_function(
            func_name=func, 
            model_info=info,
            arch_name=config_name,
            t_list=HYPERPARAMS['t_steps'], 
            schedules=HYPERPARAMS['schedules'], 
            t_start_ratios=HYPERPARAMS['t_start_ratios'], 
            lr_list=[info['lr']], 
            batch_sizes=HYPERPARAMS['batch_sizes'], 
            skip_steps_list=HYPERPARAMS['skip_steps'],
            max_epochs=TRAIN_PARAMS['max_epochs'], 
            eval_every=TRAIN_PARAMS['eval_every'], 
            patience=TRAIN_PARAMS['patience'], 
            previous_results=previous_results)

        # 3. ZAPIS FINALNY 
        with open(cache_path, 'wb') as f:
            pickle.dump(results, f)
        
        master_results[func][config_name] = results['best_metrics']


|                     ROZPOCZYNAM EKSPERYMENT II (SDEDIT)                      |

>>> ANALIZA FUNKCJI: SQUARE_WAVE
  [DONE]    UNet_C128_5e-4      : Wszystkie próby (6) gotowe.
  [DONE]    UNet_C256_5e-4      : Wszystkie próby (6) gotowe.
  [DONE]    Conv1D_C128_1e-3    : Wszystkie próby (6) gotowe.
  [DONE]    Conv1D_C128_5e-4    : Wszystkie próby (6) gotowe.
  [DONE]    MLP_C256_1e-3       : Wszystkie próby (6) gotowe.
  [DONE]    MLP_C256_5e-4       : Wszystkie próby (6) gotowe.
  [DONE]    UNet_C64_1e-4       : Wszystkie próby (6) gotowe.
  [DONE]    Conv1D_C32_5e-4     : Wszystkie próby (6) gotowe.
  [DONE]    MLP_C32_1e-4        : Wszystkie próby (6) gotowe.
  [DONE]    MLP_C256_1e-4       : Wszystkie próby (6) gotowe.

>>> ANALIZA FUNKCJI: DAMPED_OSCILLATOR
  [DONE]    UNet_C128_5e-4      : Wszystkie próby (6) gotowe.
  [DONE]    UNet_C256_5e-4      : Wszystkie próby (6) gotowe.
  [DONE]    Conv1D_C128_1e-3    : Wszystkie próby (6) gotowe.
  [DONE]    Conv1D_C128_5e-4    : Wszy

### 3. Wnioski

* Multi-Layer Perceptron (MLP) radzi sobie dobrze z prostymi, gładkimi funkcjami o niskiej częstotliwości (np. funkcje liniowe, proste wielomiany). Ale ze względu na brak zdolności do wychwytywania zależności przestrzennych (lokalnych), MLP z reguły słabiej radzi sobie z funkcjami okresowymi, mocno oscylującymi oraz sygnałami z ostrymi, nagłymi skokami. Ma tendencję do nadmiernego "wygładzania" złożonych detali.

* Sieci konwolucyjne (Conv1D) lepiej odszumia funkcje okresowe (np. sinus, kosinus) oraz sygnały posiadającymi powtarzalne, lokalne wzorce. Filtry konwolucyjne dobrze wychwytują lokalne zmiany kształtu i dynamikę sygnału (wysokie częstotliwości). Jednak może mieć problemy z uchwyceniem globalnego kontekstu (całokształtu bardzo rozciągniętej funkcji), jeśli tzw. pole recepcyjne (receptive field) modelu nie jest wystarczająco duże.

* U-Net 1D jest architekturą najbardziej uniwersalną i zazwyczaj najskuteczniejszą. Doskonale radzi sobie z funkcjami złożonymi, które łączą różne częstotliwości. Dzięki swojej strukturze (koder-dekoder) oraz połączeniom omijającym (*skip connections*), U-Net potrafi zrekonstruować zarówno drobne, lokalne detale zaszumionego sygnału, jak i bardzo dobrze zachować jego globalny, bazowy kształt. Zdecydowanie najlepiej sprawdza się w trudnych zadaniach odszumiania.

U-Net 1D to model najbardziej kompletny do zadań rekonstrukcji sygnałów, Conv1D to dobry wybór do lokalnych wzorców i oscylacji, natomiast MLP warto stosować tylko do najprostszych, bardzo gładkich przebiegów matematycznych.

In [5]:
import os
import pickle
import numpy as np
import pandas as pd
import torch

analysis_dir = '../images/experiment2/analysis'
os.makedirs(analysis_dir, exist_ok=True)

print(f"\n{'='*80}")
print(f"| {'ROZPOCZYNAM GENEROWANIE ANALIZY WIZUALNEJ (EKSPERYMENT II)':^76} |")
print(f"{'='*80}")



def ensure_3d_tensor(x, device):
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x).float()
    if not torch.is_tensor(x):
        x = torch.tensor(x).float()

    x = x.to(device)

    while x.dim() > 3 and x.shape[0] == 1:
        x = x.squeeze(0)
    while x.dim() > 3 and x.shape[1] == 1:
        x = x.squeeze(1)

    if x.dim() == 1:
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.dim() == 2:
        x = x.unsqueeze(1)
    elif x.dim() == 3:
        pass
    elif x.dim() == 4:
        if x.shape[1] == 1:
            x = x.squeeze(1)
        elif x.shape[2] == 1:
            x = x.squeeze(2)
    else:
        raise ValueError(f"Unsupported tensor shape: {x.shape}")

    assert x.dim() == 3, f"Tensor is not 3D: {x.shape}"
    assert x.shape[1] == 1, f"Channel dim must be 1: {x.shape}"

    return x


def load_best_model(config_name, func, best_cfg, runner, architectures_config):

    n_T = best_cfg['T']
    schedule = best_cfg['schedule']

    ckpt_path = None

    for run_id in range(runner.num_runs):

        filename = (
            f"{config_name}_{func}_T{n_T}_{schedule}"
            f"_run{run_id}_best_model.pth"
        )

        possible_path = os.path.join(
            runner.checkpoints_dir,
            filename
        )

        if os.path.exists(possible_path):
            ckpt_path = possible_path
            break

    if ckpt_path is None:
        return None

    model_class = architectures_config[config_name]['class']
    capacity = architectures_config[config_name]['capacity']

    try:
        model = model_class(
            data_dim=128,
            base_channels=capacity
        ).to(runner.device)

    except TypeError:
        model = model_class(
            data_dim=128,
            hidden_dim=capacity
        ).to(runner.device)

    checkpoint = torch.load(
        ckpt_path,
        map_location=runner.device,
        weights_only=False
    )

    raw_state = checkpoint['model_state_dict']

    clean_state = {}

    for k, v in raw_state.items():

        if k.startswith('_orig_mod.'):
            clean_state[k.replace('_orig_mod.', '')] = v
        else:
            clean_state[k] = v

    model.load_state_dict(clean_state)

    model.eval()

    betas = get_beta_schedule(
        best_cfg['schedule'],
        1e-4,
        0.02,
        best_cfg['T']
    )

    ddpm = DDPM1D(
        model,
        betas,
        best_cfg['T'],
        runner.device
    )

    return ddpm


def reconstruct_signal(ddpm, y_true_tensor, t_start):

    device = y_true_tensor.device

    t_tensor = torch.full(
        (1,),
        t_start - 1,
        dtype=torch.long,
        device=device
    )



    y_noisy_tensor = ddpm.q_sample(x_start=y_true_tensor, t=t_tensor)
    if y_noisy_tensor.dim() == 2:
        y_noisy_tensor = y_noisy_tensor.unsqueeze(1)
    
    assert y_noisy_tensor.dim() == 3
    current_y = y_noisy_tensor.clone()
    with torch.no_grad():
        for i in reversed(range(t_start)):
            t_batch = torch.full((1,), i, dtype=torch.long, device=device)
    
            assert current_y.dim() == 3, f"BAD SHAPE BEFORE MODEL: {current_y.shape}"
    
            # Bezpieczne przekazanie pełnego tensora 3D [B, 1, L] do modelu
            noise_pred = ddpm.model(current_y, t_batch)
            
            # Pancerne spłaszczenie nadmiarowych wymiarów 1 (np. z [1, 1, 1, 128] lub [1, 1, 128] do [128])
            # i przywrócenie kanonicznego formatu batcha [1, 1, 128]
            noise_pred = noise_pred.squeeze()
            if noise_pred.dim() == 1:
                noise_pred = noise_pred.unsqueeze(0).unsqueeze(0)
            elif noise_pred.dim() == 2:
                noise_pred = noise_pred.unsqueeze(1)
    
            assert noise_pred.shape == current_y.shape, (
                f"SHAPE MISMATCH AFTER FIX: "
                f"{noise_pred.shape} vs {current_y.shape}"
            )
# # ... (reszta funkcji reconstruct_signal z obliczeniami alpha/beta bez zmian)
#     with torch.no_grad():
#         for i in reversed(range(t_start)):
#             t_batch = torch.full((1,), i, dtype=torch.long, device=device)
    
#             assert current_y.dim() == 3, f"BAD SHAPE BEFORE MODEL: {current_y.shape}"
    
#             noise_pred = ddpm.model(current_y[:, 0, :], t_batch)      # [B,1,L] -> [B,L]
#             noise_pred = noise_pred.unsqueeze(1)                        # [B,L]  -> [B,1,L]
    

#             assert noise_pred.shape == current_y.shape, (
#                 f"SHAPE MISMATCH: "
#                 f"{noise_pred.shape} vs {current_y.shape}"
#             )

            alpha_t = ddpm.alphas[i].item()
            alpha_bar_t = ddpm.alphas_bar[i].item()
            beta_t = ddpm.betas[i].item()

            if i > 0:
                noise = torch.randn_like(current_y)
            else:
                noise = torch.zeros_like(current_y)

            current_y = (
                (1 / np.sqrt(alpha_t))
                * (
                    current_y
                    - (
                        ((1 - alpha_t) / np.sqrt(1 - alpha_bar_t))
                        * noise_pred
                    )
                )
            )

            current_y = current_y + np.sqrt(beta_t) * noise

    return y_noisy_tensor, current_y




for func in test_functions:

    print(f"\n>>> Funkcja: {func.upper()}")

    for config_name, info in architectures_config.items():

        cache_file = os.path.join(
            CACHE_DIR,
            f"results_cache_{config_name}_{func}.pkl"
        )

        if not os.path.exists(cache_file):

            print(f"  [SKIP] {config_name:20} (brak cache)")
            continue

        print(f"  [PROCESS] {config_name:20}", end=" ")

        paths = {
            'trajectory':
                os.path.join(
                    analysis_dir,
                    f"B_Trajectory_{config_name}_{func}.png"
                ),

            'heatmap':
                os.path.join(
                    analysis_dir,
                    f"C_Heatmap_{config_name}_{func}.png"
                ),

            'fft':
                os.path.join(
                    analysis_dir,
                    f"E_FFT_{config_name}_{func}.png"
                ),

            'pointwise':
                os.path.join(
                    analysis_dir,
                    f"F_PointwiseErr_{config_name}_{func}.png"
                ),

            'time_qual':
                os.path.join(
                    analysis_dir,
                    f"G_Time_vs_Qual_{config_name}_{func}.png"
                ),

            'reconstruction':
                os.path.join(
                    analysis_dir,
                    f"J_Reconstruction_{config_name}_{func}.png"
                )
        }



        with open(cache_file, 'rb') as f:
            saved_results = pickle.load(f)



        flat_history = []

        for trial in saved_results['trials']:

            params = trial['params']

            mses = []
            times = []

            for run in trial['runs']:

                metrics = run.get('all_metrics', {})

                mses.append(
                    metrics.get(
                        'MSE',
                        run.get('best_reconstruction_mse', 0)
                    )
                )

                times.append(
                    metrics.get(
                        'Sample_Time_s',
                        0.0
                    )
                )

            flat_history.append({
                'T': params['T'],
                'schedule': params['schedule'],
                'lr': params['lr'],
                'batch_size': params['batch_size'],
                'MSE': np.mean(mses),
                'exec_time': np.mean(times)
            })

        history_df = pd.DataFrame(flat_history)

        best_metrics = saved_results['best_metrics']
        best_cfg = best_metrics['params']


        x_val, y_true_np = runner.math_funcs.get_dataset(
            func,
            num_samples=1,
            mode='test'
        )

        x_val = torch.from_numpy(x_val).float().to(runner.device)

        y_true_np = np.squeeze(y_true_np)          # (128,) lub (N,128)
        y_true_tensor = ensure_3d_tensor(y_true_np, runner.device)   # -> [1,1,128]


        ddpm = load_best_model(
            config_name,
            func,
            best_cfg,
            runner,
            architectures_config
        )

        if ddpm is None:

            print("[BRAK CHECKPOINTU]")
            continue


        if not os.path.exists(paths['heatmap']):
            vis.plot_hyperparameter_heatmaps(
                history_df,
                func,
                config_name,
                paths['heatmap']
            )

        if not os.path.exists(paths['time_qual']):
            vis.plot_time_vs_quality(
                history_df,
                func,
                config_name,
                paths['time_qual']
            )



        t_start = max(
            1,
            int(0.5 * best_cfg['T'])
        )

        y_noisy_tensor, y_pred_tensor = reconstruct_signal(
            ddpm,
            y_true_tensor,
            t_start
        )

        y_true = y_true_tensor[0, 0].cpu().numpy()
        y_noisy = y_noisy_tensor[0, 0].cpu().numpy()
        y_pred = y_pred_tensor[0, 0].cpu().numpy()



        if not os.path.exists(paths['reconstruction']):

            vis.plot_noisy_reconstruction(
                x_val[0].cpu().numpy(),
                y_true,
                y_noisy,
                y_pred,
                func,
                config_name,
                paths['reconstruction']
            )

        if not os.path.exists(paths['fft']):

            vis.plot_fft_spectrum(
                x_val[0].cpu().numpy(),
                y_true,
                y_pred,
                func,
                config_name,
                paths['fft']
            )

        if not os.path.exists(paths['pointwise']):

            vis.plot_pointwise_error(
                x_val[0].cpu().numpy(),
                y_true,
                y_pred,
                func,
                config_name,
                paths['pointwise'],
                config=best_cfg,
                metrics=best_metrics
            )

        if not os.path.exists(paths['trajectory']):

            vis.plot_denoising_trajectory(
                ddpm,
                x_val[0].cpu().numpy(),
                y_true,
                t_start,
                runner.device,
                paths['trajectory'],
                config=best_cfg,
                metrics=best_metrics,
                arch_name=config_name,
                func_name=func
            )

        print("-> [DONE]")

print(f"\n{'='*80}")
print(f"| {'PROCES WIZUALIZACJI ZAKOŃCZONY POMYŚLNIE':^76} |")
print(f"{'='*80}")


|          ROZPOCZYNAM GENEROWANIE ANALIZY WIZUALNEJ (EKSPERYMENT II)          |

>>> Funkcja: SQUARE_WAVE
  [PROCESS] UNet_C128_5e-4       -> [DONE]
  [PROCESS] UNet_C256_5e-4       -> [DONE]
  [PROCESS] Conv1D_C128_1e-3     -> [DONE]
  [PROCESS] Conv1D_C128_5e-4     -> [DONE]
  [PROCESS] MLP_C256_1e-3        -> [DONE]
  [PROCESS] MLP_C256_5e-4        -> [DONE]
  [PROCESS] UNet_C64_1e-4        -> [DONE]
  [PROCESS] Conv1D_C32_5e-4      -> [DONE]
  [PROCESS] MLP_C32_1e-4         -> [DONE]
  [PROCESS] MLP_C256_1e-4        -> [DONE]

>>> Funkcja: DAMPED_OSCILLATOR
  [PROCESS] UNet_C128_5e-4       -> [DONE]
  [PROCESS] UNet_C256_5e-4       -> [DONE]
  [PROCESS] Conv1D_C128_1e-3     -> [DONE]
  [PROCESS] Conv1D_C128_5e-4     -> [DONE]
  [PROCESS] MLP_C256_1e-3        -> [DONE]
  [PROCESS] MLP_C256_5e-4        -> [DONE]
  [PROCESS] UNet_C64_1e-4        -> [DONE]
  [PROCESS] Conv1D_C32_5e-4      -> [DONE]
  [PROCESS] MLP_C32_1e-4         -> [DONE]
  [PROCESS] MLP_C256_1e-4        -> [DONE]


In [6]:
# 1. Agregacja (załóżmy, że jesteś już po pętli wszystkich eksperymentów)
df_metrics = aggregate_experiment_metrics(test_functions, architectures_config, CACHE_DIR)

# 2. Tabele w folderze z analizą
generate_comparison_tables(df_metrics, analysis_dir)

# 3. Generowanie wykresów porównawczych
for func in test_functions:
    # Ranking SNR (Więcej = lepiej, dlatego ascending=False)
    vis.plot_metric_bar_comparison(
        df_metrics, 
        func, 
        metric='SNR', 
        ascending=False, 
        save_path=os.path.join(analysis_dir, f"K_Ranking_SNR_{func}.png")
    )
    
    # Ranking L2 Error (Mniej = lepiej, dlatego ascending=True)
    vis.plot_metric_bar_comparison(
        df_metrics, 
        func, 
        metric='L2_Error', 
        ascending=True, 
        save_path=os.path.join(analysis_dir, f"K_Ranking_L2_{func}.png")
    )
    
    # Wykres Radarowy. Podajemy nazwy modeli, które chcemy ze sobą zderzyć
    # Wybieram np. po jednym najlepszym z UNet, Conv1D, MLP + najgorszy z MLP
    models_to_compare = [
        'UNet_C128_5e-4',     # Najlepszy UNet
        'Conv1D_C128_1e-3',   # Najlepszy Conv1D
        'MLP_C256_1e-3',      # Najlepszy MLP
        'MLP_C32_1e-4'        # Przypadek anomalny (Underfitting)
    ]
    
    vis.plot_radar_metrics_comparison(
        df_metrics, 
        func, 
        architectures_to_compare=models_to_compare,
        save_path=os.path.join(analysis_dir, f"L_Radar_Plot_{func}.png")
    )


[SQUARE_WAVE] ANALIZA METRYK:
 -> Najlepszy model (najwyższe SNR): UNet_C256_5e-4 (39.33 dB)
 -> Najgorszy model (najniższe SNR): MLP_C32_1e-4 (15.06 dB)

[DAMPED_OSCILLATOR] ANALIZA METRYK:
 -> Najlepszy model (najwyższe SNR): Conv1D_C128_1e-3 (25.89 dB)
 -> Najgorszy model (najniższe SNR): MLP_C32_1e-4 (6.35 dB)

[MIXED_FREQ] ANALIZA METRYK:
 -> Najlepszy model (najwyższe SNR): Conv1D_C128_1e-3 (30.10 dB)
 -> Najgorszy model (najniższe SNR): MLP_C32_1e-4 (9.83 dB)


AttributeError: module 'utils.visualisations' has no attribute 'plot_radar_metrics_comparison'

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_and_plot_best_architectures(test_functions, architectures_config, cache_dir='experiments/cache', save_dir='../images/experiment2/analysis'):
    """
    Automatycznie wyciąga najlepsze konfiguracje per klasa architektury,
    sortuje je w kolejności [MLP, Conv1D, UNet] i generuje wykresy porównawcze
    z bezpiecznym marginesem pionowym osi OY.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    def get_arch_type(name):
        if 'UNet' in name: return 'UNet'
        if 'Conv1D' in name: return 'Conv1D'
        if 'MLP' in name: return 'MLP'
        return 'Inna'

    for func in test_functions:
        print(f"\n{'='*60}\nKompilacja wyników dla funkcji: {func.upper()}\n{'='*60}")
        
        best_per_arch = {}
        
        for config_name in architectures_config.keys():
            cache_file = os.path.join(cache_dir, f"results_cache_{config_name}_{func}.pkl")
            if not os.path.exists(cache_file):
                continue
                
            with open(cache_file, 'rb') as f:
                saved_results = pickle.load(f)
                
            arch_type = get_arch_type(config_name)
            
            best_trial_mse = float('inf')
            best_trial_params = None
            
            for trial in saved_results['trials']:
                mses = [run.get('all_metrics', {}).get('MSE', run.get('best_reconstruction_mse', float('inf'))) for run in trial['runs']]
                mean_mse = np.mean(mses)
                
                if mean_mse < best_trial_mse:
                    best_trial_mse = mean_mse
                    best_trial_params = trial['params']
            
            if best_trial_params is not None:
                if arch_type not in best_per_arch or best_trial_mse < best_per_arch[arch_type]['MSE']:
                    best_per_arch[arch_type] = {
                        'Nazwa konfiguracji': config_name,
                        'MSE': best_trial_mse,
                        'Optymalne T': best_trial_params['T'],
                        'Harmonogram szumu': best_trial_params['schedule'],
                        'Głębokość SDEdit (Ratio)': best_trial_params.get('t_start_ratio', 'N/A')
                    }
                    
        if not best_per_arch:
            print(f"[UWAGA] Brak danych w cache dla funkcji {func}. Pomijam.")
            continue
            
        # Tworzenie DataFrame
        df_summary = pd.DataFrame.from_dict(best_per_arch, orient='index').reset_index()
        df_summary.rename(columns={'index': 'Klasa architektury'}, inplace=True)
        
        # --- POPRAWKA 1: Wymuszenie rygorystycznej kolejności [MLP, Conv1D, UNet] ---
        df_summary['Klasa architektury'] = pd.Categorical(
            df_summary['Klasa architektury'], 
            categories=['MLP', 'Conv1D', 'UNet'], 
            ordered=True
        )
        df_summary = df_summary.sort_values('Klasa architektury').reset_index(drop=True)
        
        print(f"\n--- Najlepsze konfiguracje architektur dla funkcji {func.upper()} ---")
        display(df_summary.style.highlight_min(subset=['MSE'], color='#d4edda').format({'MSE': '{:.2e}'}))
        
        # Generowanie wykresu
        custom_rc = {
            'figure.autolayout': False,
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
        }
        
        with plt.rc_context(custom_rc):
            fig, ax = plt.subplots(figsize=(12 / 2.54, 7.5 / 2.54))
            
            # Rysowanie słupków z zachowaniem posortowanej kolejności kategorycznej
            sns.barplot(
                data=df_summary, 
                x='Klasa architektury', 
                y='MSE', 
                palette=['#9467bd', '#2ca02c', '#1f77b4'], # Zmiana kolejności barw dopasowana do etykiet
                edgecolor='black', 
                linewidth=0.7, 
                ax=ax
            )
            
            # Wyznaczenie parametrów skali i maksimów danych
            max_mse = df_summary['MSE'].max()
            min_mse = df_summary['MSE'].min()
            
            use_log = (max_mse / min_mse > 10) or (min_mse < 0.001)
            
            # --- POPRAWKA 2: Dynamiczne zwiększenie zakresu osi OY (Zapas na etykiety) ---
            if use_log:
                ax.set_yscale('log')
                ax.set_ylabel('Błąd średniokwadratowy MSE (log)', fontsize=10)
                # Dajemy zapas mnożnikowy w skali logarytmicznej (podnosimy sufit o pół rzędu wielkości)
                ax.set_ylim(bottom=min_mse * 0.5, top=max_mse * 4.5)
            else:
                ax.set_ylabel('Błąd średniokwadratowy MSE', fontsize=10)
                # Dajemy 35% sztywnego zapasu w skali liniowej nad najwyższym słupkiem
                ax.set_ylim(0, max_mse * 1.35)
                
            # Dodanie wartości tekstowych nad słupkami z kontrolowanym offsetem
            for p in ax.patches:
                val = p.get_height()
                if pd.notna(val) and val > 0:
                    label_text = f'{val:.2e}' if val < 0.01 else f'{val:.4f}'
                    ax.annotate(label_text, 
                                (p.get_x() + p.get_width() / 2., val), 
                                ha='center', va='bottom', 
                                xytext=(0, 4), # Podniesienie napisu o 4 punkty typograficzne nad słupek
                                textcoords='offset points', 
                                fontsize=8.5, fontweight='semibold')
            
            ax.set_title(f'Porównanie klas architektur dyfuzyjnych SDEdit\nFunkcja testowa: {func.upper()}', 
                         pad=14, fontsize=11, fontweight='bold')
            ax.set_xlabel('Rozpatrywana klasa architektury sieciowej', fontsize=10, labelpad=8)
            ax.tick_params(axis='both', labelsize=9)
            ax.grid(True, linestyle='--', alpha=0.4, axis='y', which="both")
            
            plt.subplots_adjust(left=0.16, right=0.94, bottom=0.18, top=0.84)
            
            save_path = os.path.join(save_dir, f"best_arch_comparison_{func.lower()}.png")
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.show()
            print(f"Zapisano wykres porównawczy: {save_path}\n")

# Wywołanie skryptu
analyze_and_plot_best_architectures(test_functions, architectures_config)

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_global_sdedit_report(test_functions, architectures_config, cache_dir='experiments/cache', save_dir='../images/experiment2/analysis'):
    os.makedirs(save_dir, exist_ok=True)
    
    all_records = []
    
    # 1. AGREGACJA WSZYSTKICH PRÓB ZE WSZYSTKICH PLIKÓW CACHE
    for func in test_functions:
        for config_name in architectures_config.keys():
            cache_file = os.path.join(cache_dir, f"results_cache_{config_name}_{func}.pkl")
            if not os.path.exists(cache_file):
                continue
                
            with open(cache_file, 'rb') as f:
                saved_results = pickle.load(f)
                
            for trial in saved_results['trials']:
                params = trial['params']
                mses = [run.get('all_metrics', {}).get('MSE', run.get('best_reconstruction_mse', float('inf'))) for run in trial['runs']]
                times = [run.get('all_metrics', {}).get('Sample_Time_s', 0.0) for run in trial['runs']]
                
                # Budujemy pełną bazę pojedynczych konfiguracji
                all_records.append({
                    'Funkcja': func.upper(),
                    'Architektura': config_name,
                    'T': params['T'],
                    'Schedule': params['schedule'],
                    'Ratio': params.get('t_start_ratio', params.get('t_start_ratios', 0.35)), # obsługa różnych wersji klucza
                    'MSE': np.mean(mses),
                    'Czas': np.mean(times)
                })
                
    if not all_records:
        print("[BŁĄD] Brak danych w cache do wygenerowania raportu globalnego.")
        return
        
    df_global = pd.DataFrame(all_records)
    
    # =========================================================================
    # RAPORT 1: NAJLEPSZE PARAMETRY DLA KAŻDEJ FUNKCJI (Dedykowany wybór)
    # =========================================================================
    print(f"\n{'='*80}\n| RAPORT 1: OPTYMALNE JEDNOSTKOWE KONFIGURACJE DLA KAŻDEJ FUNKCJI |\n{'='*80}")
    
    best_per_function = df_global.loc[df_global.groupby('Funkcja')['MSE'].idxmin()].reset_index(drop=True)
    display(best_per_function.style.format({'MSE': '{:.2e}', 'Czas': '{:.2f}s'}))
    
    # Zapis do CSV (przydatne do szybkiego stworzenia tabeli w LaTeX)
    best_per_function.to_csv(os.path.join(save_dir, 'sdedit_best_per_function.csv'), index=False)
    
    # =========================================================================
    # RAPORT 2: POSZUKIWANIE "ZŁOTEGO STRZAŁU" (Analiza Makroskopowa)
    # =========================================================================
    print(f"\n{'='*80}\n| RAPORT 2: GLOBALNY RANKING PARAMETRÓW (W POSZUKIWANIU ZŁOTEGO STRZAŁU) |\n{'='*80}")
    
    # Grupujemy po parametrach i obliczamy średni błąd oraz medianę błędu dla całego eksperymentu
    df_params_ranking = df_global.groupby(['T', 'Schedule', 'Ratio']).agg(
        Sredni_Blad_MSE=('MSE', 'mean'),
        Mediana_Bledu_MSE=('MSE', 'median'),
        Sredni_Czas_Inferencji=('Czas', 'mean'),
        Liczba_Sukcesow=('MSE', 'count') # ile razy dana konfiguracja zbiegła
    ).reset_index()
    
    # Sortujemy po medianie błędu (mediana jest odporna na anomalie/eksplozje pojedynczych funkcji)
    df_params_ranking = df_params_ranking.sort_values('Mediana_Bledu_MSE').reset_index(drop=True)
    
    print("\nTOP 5 GLOBALNYCH KONFIGURACJI (Najbardziej uniwersalne zestawy parametrów):")
    display(df_params_ranking.head(5).style.format({
        'Sredni_Blad_MSE': '{:.2e}', 
        'Mediana_Bledu_MSE': '{:.2e}', 
        'Sredni_Czas_Inferencji': '{:.2f}s'
    }))
    

    custom_rc = {
        'figure.autolayout': False,
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
    }
    
    with plt.rc_context(custom_rc):
        # Pivot table do zweryfikowania wpływu liczby kroków T oraz głębokości zaszumienia (Ratio)
        df_pivot = df_global.groupby(['Ratio', 'T'])['MSE'].median().unstack()
        
        fig, ax = plt.subplots(figsize=(13 / 2.54, 9 / 2.54))
        
        # Tworzymy macierz etykiet tekstowych w notacji naukowej
        annot_labels = np.zeros_like(df_pivot.values, dtype=object)
        for r in range(df_pivot.shape[0]):
            for c in range(df_pivot.shape[1]):
                annot_labels[r, c] = f"{df_pivot.values[r,c]:.1e}"
        
        sns.heatmap(
            df_pivot, 
            annot=annot_labels, 
            fmt="", 
            cmap='viridis_r', 
            linewidths=0.8, 
            linecolor='white',
            cbar_kws={'label': 'Mediana globalnego błędu MSE'},
            ax=ax,
            annot_kws={'size': 8.5, 'weight': 'semibold'}
        )
        
        ax.set_title('Globalna mapa wrażliwości hiperparametrycznej SDEdit\n(Zagregowane wyniki dla wszystkich funkcji testowych)', 
                     pad=14, fontsize=11, fontweight='bold')
        ax.set_ylabel('Głębokość zaszumienia startowego (t_start_ratio)', fontsize=10, labelpad=10)
        ax.set_xlabel('Całkowita liczba kroków dyskretyzacji czasowej (T)', fontsize=10, labelpad=10)
        
        plt.subplots_adjust(left=0.16, right=0.84, bottom=0.16, top=0.84)
        
        save_path = os.path.join(save_dir, 'global_sdedit_hyperparameter_heatmap.png')
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close()
        print(f"\nWygenerowano i zapisano globalną mapę ciepła parametrów: {save_path}")

# Uruchomienie globalnego silnika raportującego
generate_global_sdedit_report(test_functions, architectures_config)

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

def generate_comprehensive_global_report(test_functions, architectures_config, runner, cache_dir='experiments/cache', save_dir='../images/experiment2/analysis'):
    """
    Agreguje globalnie wszystkie metryki (MSE, L2, Wasserstein, Pearson, Czas) ze wszystkich funkcji testowych.
    Wyłania najlepszą i najgorszą konfigurację, a następnie generuje wykres porównawczy odszumiania.
    """
    os.makedirs(save_dir, exist_ok=True)
    all_records = []
    
    print(">>> Krok 1: Agregacja danych z cache...")
    for func in test_functions:
        for config_name in architectures_config.keys():
            cache_file = os.path.join(cache_dir, f"results_cache_{config_name}_{func}.pkl")
            if not os.path.exists(cache_file):
                continue
                
            with open(cache_file, 'rb') as f:
                saved_results = pickle.load(f)
                
            for trial in saved_results['trials']:
                params = trial['params']
                
                # Bezpieczne wyciąganie poszczególnych metryk z list przebiegów (runs)
                mses = [run.get('all_metrics', {}).get('MSE', run.get('best_reconstruction_mse', float('inf'))) for run in trial['runs']]
                l2s = [run.get('all_metrics', {}).get('L2_Error', run.get('all_metrics', {}).get('L2', float('inf'))) for run in trial['runs']]
                wassersteins = [run.get('all_metrics', {}).get('Wasserstein_Distance', run.get('all_metrics', {}).get('Wasserstein', float('inf'))) for run in trial['runs']]
                pearsons = [run.get('all_metrics', {}).get('Pearson_Correlation', run.get('all_metrics', {}).get('Pearson', 0.0)) for run in trial['runs']]
                times = [run.get('all_metrics', {}).get('Sample_Time_s', 0.0) for run in trial['runs']]
                
                all_records.append({
                    'Funkcja': func.upper(),
                    'Architektura': config_name,
                    'T': params['T'],
                    'Schedule': params['schedule'],
                    'Ratio': params.get('t_start_ratio', 0.35),
                    'MSE': np.mean(mses),
                    'L2_Error': np.mean(l2s),
                    'Wasserstein': np.mean(wassersteins),
                    'Pearson': np.mean(pearsons),
                    'Czas_Inferencji_s': np.mean(times)
                })
                
    if not all_records:
        print("[BŁĄD] Brak danych w cache. Upewnij się, że eksperymenty zostały zapisane.")
        return None, None
        
    df_global = pd.DataFrame(all_records)
    
    # Globalne uśrednienie parametrów po wszystkich funkcjach (szukamy konfiguracji uniwersalnej)
    df_ranking = df_global.groupby(['Architektura', 'T', 'Schedule', 'Ratio']).agg(
        Mean_MSE=('MSE', 'mean'),
        Median_MSE=('MSE', 'median'),
        Mean_L2=('L2_Error', 'mean'),
        Mean_Wasserstein=('Wasserstein', 'mean'),
        Mean_Pearson=('Pearson', 'mean'),
        Mean_Inference_Time=('Czas_Inferencji_s', 'mean')
    ).reset_index()
    
    # Sortujemy po medianie błędu MSE (odporna na anomalie eksplozji gradientu w pojedynczych konfiguracjach)
    df_ranking = df_ranking.sort_values('Median_MSE').reset_index(drop=True)
    
    print(f"\n{'='*90}\n| GLOBALNY RANKING KONFIGURACJI PARAMETRYCZNYCH |\n{'='*90}")
    print("\nTOP 3 NAJLEPSZE GLOBALNIE KONFIGURACJE:")
    display(df_ranking.head(3).style.format({
        'Mean_MSE': '{:.2e}', 'Median_MSE': '{:.2e}', 'Mean_L2': '{:.4f}', 
        'Mean_Wasserstein': '{:.4f}', 'Mean_Pearson': '{:.4f}', 'Mean_Inference_Time': '{:.4f}s'
    }))
    
    print("\nTOP 3 NAJGORSZE GLOBALNIE KONFIGURACJE:")
    display(df_ranking.tail(3).style.format({
        'Mean_MSE': '{:.2e}', 'Median_MSE': '{:.2e}', 'Mean_L2': '{:.4f}', 
        'Mean_Wasserstein': '{:.4f}', 'Mean_Pearson': '{:.4f}', 'Mean_Inference_Time': '{:.4f}s'
    }))
    
    # Wyciągamy parametry najlepszej i najgorszej konfiguracji
    best_config_row = df_ranking.iloc[0]
    worst_config_row = df_ranking.iloc[-1]
    
    # Zapis bazy do pliku csv
    df_ranking.to_csv(os.path.join(save_dir, 'global_sdedit_complexity_quality_summary.csv'), index=False)
    
    return best_config_row, worst_config_row

def plot_best_vs_worst_comparison(runner, test_functions, architectures_config, best_cfg, worst_cfg, selected_func='square_wave', save_dir='../images/experiment2/analysis'):
    """
    Pobiera i przygotowuje dane, ładuje wagi modeli dyfuzyjnych, przeprowadza pancernie bezpieczną 
    rekonstrukcję sygnału sygnału i zestawia wyniki na jednym, czytelnym wykresie.
    """
    print(f"\n>>> Krok 2: Generowanie porównania graficznego dla funkcji: {selected_func.upper()}...")
    
    # 1. Pobranie oryginalnego sygnału testowego
    x_val, y_true_raw = runner.math_funcs.get_dataset(selected_func, num_samples=1, mode='test')
    x_axis = x_val[0]
    y_true_np = np.squeeze(y_true_raw)
    
    # Konwersja do bezpiecznego formatu 3D [1, 1, 128]
    y_true_tensor = ensure_3d_tensor(y_true_np, runner.device)
    
    # 2. Ładowanie najlepszego i najgorszego modelu dyfuzyjnego
    best_params = {'T': best_cfg['T'], 'schedule': best_cfg['Schedule']}
    worst_params = {'T': worst_cfg['T'], 'schedule': worst_cfg['Schedule']}
    
    ddpm_best = load_best_model(best_cfg['Architektura'], selected_func, best_params, runner, architectures_config)
    ddpm_worst = load_best_model(worst_cfg['Architektura'], selected_func, worst_params, runner, architectures_config)
    
    if ddpm_best is None or ddpm_worst is None:
        print("[BŁĄD] Nie można załadować punktów kontrolnych (checkpoints) dla wyznaczonych modeli!")
        return
        
    # 3. Rekonstrukcja sygnału - Najlepszy Model
    t_start_best = max(1, int(best_cfg['Ratio'] * best_cfg['T']))
    y_noisy_best_tensor, y_pred_best_tensor = reconstruct_signal_safe(ddpm_best, y_true_tensor, t_start_best)
    
    # 4. Rekonstrukcja sygnału - Najgorszy Model
    t_start_worst = max(1, int(worst_cfg['Ratio'] * worst_cfg['T']))
    y_noisy_worst_tensor, y_pred_worst_tensor = reconstruct_signal_safe(ddpm_worst, y_true_tensor, t_start_worst)
    
    # Wyciąganie wektorów do wykresów (płaskie tablice numpy)
    y_true_plot = y_true_tensor[0, 0].cpu().numpy()
    y_noisy_best_plot = y_noisy_best_tensor[0, 0].cpu().numpy()
    y_noisy_worst_plot = y_noisy_worst_tensor[0, 0].cpu().numpy()
    y_pred_best_plot = y_pred_best_tensor[0, 0].cpu().numpy()
    y_pred_worst_plot = y_pred_worst_tensor[0, 0].cpu().numpy()
    
    # 5. Tworzenie profesjonalnego, podwójnego wykresu porównawczego
    custom_rc = {
        'figure.autolayout': False,
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
    }
    
    with plt.rc_context(custom_rc):
        fig, axes = plt.subplots(1, 2, figsize=(16 / 2.54, 7.5 / 2.54), sharey=True)
        
        # Panel Lewy: Najlepsza Konfiguracja
        axes[0].plot(x_axis, y_true_plot, label='Sygnał oryginalny', color='black', linewidth=1.5, zorder=3)
        axes[0].scatter(x_axis, y_noisy_best_plot, label='Zaszumiony (SDEdit)', color='gray', alpha=0.4, s=8)
        axes[0].plot(x_axis, y_pred_best_plot, label='Rekonstrukcja (Najlepsza)', color='#2ca02c', linewidth=1.8, linestyle='--')
        title_best = f"Najlepsza: {best_cfg['Architektura']}\n$T={best_cfg['T']}$, Sch: {best_cfg['Schedule']}, $r={best_cfg['Ratio']}$"
        axes[0].set_title(title_best, fontsize=8.5, fontweight='bold', pad=8)
        axes[0].set_xlabel('Dziedzina czasu ($x$)', fontsize=9)
        axes[0].set_ylabel('Amplituda sygnału ($y$)', fontsize=9)
        axes[0].grid(True, linestyle='--', alpha=0.5)
        axes[0].legend(fontsize=7, loc='upper right')
        
        # Panel Prawy: Najgorsza Konfiguracja
        axes[1].plot(x_axis, y_true_plot, label='Sygnał oryginalny', color='black', linewidth=1.5, zorder=3)
        axes[1].scatter(x_axis, y_noisy_worst_plot, label='Zaszumiony (SDEdit)', color='gray', alpha=0.4, s=8)
        axes[1].plot(x_axis, y_pred_worst_plot, label='Rekonstrukcja (Najgorsza)', color='#d62728', linewidth=1.8, linestyle='-.')
        title_worst = f"Najgorsza: {worst_cfg['Architektura']}\n$T={worst_cfg['T']}$, Sch: {worst_cfg['Schedule']}, $r={worst_cfg['Ratio']}$"
        axes[1].set_title(title_worst, fontsize=8.5, fontweight='bold', pad=8)
        axes[1].set_xlabel('Dziedzina czasu ($x$)', fontsize=9)
        axes[1].grid(True, linestyle='--', alpha=0.5)
        axes[1].legend(fontsize=7, loc='upper right')
        
        plt.suptitle(f"Globalne zestawienie skrajnych konfiguracji SDEdit\nFunkcja testowa: {selected_func.upper()}", 
                     fontsize=10.5, fontweight='bold', y=1.02)
        
        plt.subplots_adjust(left=0.1, right=0.95, bottom=0.15, top=0.82, wspace=0.15)
        
        save_path = os.path.join(save_dir, f"global_best_vs_worst_{selected_func.lower()}.png")
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        print(f">>> Wykres wygenerowany i zapisany w: {save_path}")

def reconstruct_signal_safe(ddpm, y_true_tensor, t_start):
    """
    Pancernie bezpieczna wersja pętli rekonstrukcyjnej SDEdit chroniąca przed 
    rozbieżnością wymiarów wyjściowych z sieci MLP/Conv1D/UNet.
    """
    device = y_true_tensor.device
    t_tensor = torch.full((1,), t_start - 1, dtype=torch.long, device=device)
    
    y_noisy_tensor = ddpm.q_sample(x_start=y_true_tensor, t=t_tensor)
    if y_noisy_tensor.dim() == 2:
        y_noisy_tensor = y_noisy_tensor.unsqueeze(1)
        
    current_y = y_noisy_tensor.clone()
    
    with torch.no_grad():
        for i in reversed(range(t_start)):
            t_batch = torch.full((1,), i, dtype=torch.long, device=device)
            
            # Wywołanie modelu niezależnie od wymaganego formatu wejściowego architektury
            noise_pred = ddpm.model(current_y, t_batch)
            
            # Wymuszenie powrotu do kanonicznej struktury [B, 1, L]
            noise_pred = noise_pred.squeeze()
            if noise_pred.dim() == 1:
                noise_pred = noise_pred.unsqueeze(0).unsqueeze(0)
            elif noise_pred.dim() == 2:
                noise_pred = noise_pred.unsqueeze(1)
                
            alpha_t = ddpm.alphas[i].item()
            alpha_bar_t = ddpm.alphas_bar[i].item()
            beta_t = ddpm.betas[i].item()
            
            if i > 0:
                noise = torch.randn_like(current_y)
            else:
                noise = torch.zeros_like(current_y)
                
            current_y = (1 / np.sqrt(alpha_t)) * (current_y - (((1 - alpha_t) / np.sqrt(1 - alpha_bar_t)) * noise_pred))
            current_y = current_y + np.sqrt(beta_t) * noise
            
    return y_noisy_tensor, current_y

# =========================================================================
# URUCHOMIENIE SILNIKA PODSUMOWUJĄCEGO I WIZUALIZACYJNEGO
# =========================================================================
# 1. Wykonanie globalnego raportu i pobranie konfiguracji granicznych
best_cfg, worst_cfg = generate_comprehensive_global_report(test_functions, architectures_config, runner)

# 2. Wyrysowanie porównania na wybranej funkcji (np. 'square_wave' z uwagi na trudne ostre krawędzie)
if best_cfg is not None and worst_cfg is not None:
    plot_best_vs_worst_comparison(runner, test_functions, architectures_config, best_cfg, worst_cfg, selected_func='damped_oscillator')

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

def generate_safe_global_report(test_functions, architectures_config, runner, cache_dir='experiments/cache', save_dir='../images/experiment2/analysis'):
    """
    Bezpieczna wersja raportu dopasowana do Twojego trwającego cache.
    Agreguje MSE, L2, Wassersteina, Korelację Pearsona oraz Czas bez ryzyka błędów klucza.
    """
    os.makedirs(save_dir, exist_ok=True)
    all_records = []
    
    print(">>> Agregacja danych z trwającego 7 dni eksperymentu...")
    for func in test_functions:
        for config_name in architectures_config.keys():
            cache_file = os.path.join(cache_dir, f"results_cache_{config_name}_{func}.pkl")
            if not os.path.exists(cache_file):
                continue
                
            with open(cache_file, 'rb') as f:
                saved_results = pickle.load(f)
                
            for trial in saved_results['trials']:
                params = trial['params']
                
                # Bezpieczne wyciąganie metryk z każdego przebiegu (run)
                mses = [run.get('all_metrics', {}).get('MSE', run.get('best_reconstruction_mse', float('inf'))) for run in trial['runs']]
                l2s = [run.get('all_metrics', {}).get('L2_Error', run.get('all_metrics', {}).get('L2', float('inf'))) for run in trial['runs']]
                wassersteins = [run.get('all_metrics', {}).get('Wasserstein_Distance', run.get('all_metrics', {}).get('Wasserstein', float('inf'))) for run in trial['runs']]
                pearsons = [run.get('all_metrics', {}).get('Pearson_Correlation', run.get('all_metrics', {}).get('Pearson', 1.0)) for run in trial['runs']]
                times = [run.get('all_metrics', {}).get('Sample_Time_s', 0.0) for run in trial['runs']]
                
                # Wyciągamy zarejestrowane ratio z all_metrics, a jeśli go tam nie ma - bezpieczny fallback
                ratios = [run.get('all_metrics', {}).get('t_start_ratio', 0.35) for run in trial['runs']]
                
                all_records.append({
                    'Funkcja': func.upper(),
                    'Architektura': config_name,
                    'T': params['T'],
                    'Schedule': params['schedule'],
                    'Ratio': np.mean(ratios), # Prawdziwe ratio przypisane do najlepszego wyniku
                    'MSE': np.mean(mses),
                    'L2_Error': np.mean(l2s),
                    'Wasserstein': np.mean(wassersteins),
                    'Pearson': np.mean(pearsons),
                    'Czas_Inferencji_s': np.mean(times)
                })
                
    if not all_records:
        print("[UWAGA] Brak plików w cache. Poczekaj na zapis pierwszych wyników.")
        return None, None
        
    df_global = pd.DataFrame(all_records)
    
    # Grupowanie i wyłonienie najlepszej/najgorszej konfiguracji na podstawie mediany MSE
    df_ranking = df_global.groupby(['Architektura', 'T', 'Schedule', 'Ratio']).agg(
        Mean_MSE=('MSE', 'mean'),
        Median_MSE=('MSE', 'median'),
        Mean_L2=('L2_Error', 'mean'),
        Mean_Wasserstein=('Wasserstein', 'mean'),
        Mean_Pearson=('Pearson', 'mean'),
        Mean_Time=('Czas_Inferencji_s', 'mean')
    ).reset_index().sort_values('Median_MSE').reset_index(drop=True)
    
    print(f"\n{'='*95}\n| GLOBALNE PODSUMOWANIE JAKOŚCI I ZŁOŻONOŚCI (ZAKOŃCZONO) |\n{'='*95}")
    print("\nTOP 3 NAJLEPSZE KONFIGURACJE W CAŁYM EKSPERYMENCIE:")
    display(df_ranking.head(3).style.format({'Mean_MSE': '{:.2e}', 'Median_MSE': '{:.2e}', 'Mean_L2': '{:.4f}', 'Mean_Wasserstein': '{:.4f}', 'Mean_Pearson': '{:.4f}', 'Mean_Time': '{:.4f}s'}))
    
    print("\nTOP 3 NAJGORSZE KONFIGURACJE W CAŁYM EKSPERYMENCIE:")
    display(df_ranking.tail(3).style.format({'Mean_MSE': '{:.2e}', 'Median_MSE': '{:.2e}', 'Mean_L2': '{:.4f}', 'Mean_Wasserstein': '{:.4f}', 'Mean_Pearson': '{:.4f}', 'Mean_Time': '{:.4f}s'}))
    
    # Zapis do CSV dla celów pracy/LaTeX
    df_ranking.to_csv(os.path.join(save_dir, 'global_report_sdedit_final.csv'), index=False)
    
    return df_ranking.iloc[0], df_ranking.iloc[-1]

# Uruchomienie kodu raportu (wywołaj w nowej komórce po zakończeniu obliczeń)
best_cfg, worst_cfg = generate_safe_global_report(test_functions, architectures_config, runner)

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_clean_thesis_plots(test_functions, architectures_config, cache_dir='experiments/cache', save_dir='../images/experiment2/analysis'):
    """
    Generuje czytelny, dwupanelowy wykres statystyczny do pracy magisterskiej.
    Rozbija dane na architekturę i kroki T, eliminując natłok na osi OX.
    """
    os.makedirs(save_dir, exist_ok=True)
    all_records = []
    
    # 1. Bezpieczna agregacja danych z cache (bez dotykania trwającego kodu)
    for func in test_functions:
        for config_name in architectures_config.keys():
            cache_file = os.path.join(cache_dir, f"results_cache_{config_name}_{func}.pkl")
            if not os.path.exists(cache_file):
                continue
                
            with open(cache_file, 'rb') as f:
                saved_results = pickle.load(f)
                
            for trial in saved_results['trials']:
                params = trial['params']
                mses = [run.get('all_metrics', {}).get('MSE', run.get('best_reconstruction_mse', float('inf'))) for run in trial['runs']]
                times = [run.get('all_metrics', {}).get('Sample_Time_s', 0.0) for run in trial['runs']]
                
                # Określenie głównej klasy architektury
                arch_type = 'UNet' if 'UNet' in config_name else ('Conv1D' if 'Conv1D' in config_name else 'MLP')
                
                all_records.append({
                    'Architektura': arch_type,
                    'T': params['T'],
                    'MSE': np.mean(mses),
                    'Czas_s': np.mean(times)
                })
                
    if not all_records:
        print("[BŁĄD] Brak danych w cache.")
        return
        
    df = pd.DataFrame(all_records)
    
    # 2. Ustawienia estetyki akademickiej (szeryfowe czcionki, wysokie DPI)
    custom_rc = {
        'figure.autolayout': False,
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif']
    }
    
    with plt.rc_context(custom_rc):
        # Szerokość 16 cm (szerokość szpalty w pracy mgr) na 7.5 cm wysokości
        fig, axes = plt.subplots(1, 2, figsize=(16 / 2.54, 7.5 / 2.54))
        
        # Paleta barw: stonowana, profesjonalna
        palette_colors = {80: '#A3B18A', 100: '#588157', 120: '#3A5A40'}
        
        # --- PANEL LEWY: Złożoność obliczeniowa (Czas) ---
        sns.boxplot(
            data=df, x='Architektura', y='Czas_s', hue='T', 
            palette=palette_colors, ax=axes[0], width=0.6, 
            linewidth=0.8, fliersize=0, boxprops=dict(alpha=0.85)
        )
        # Nałożenie punktów dla pokazania pełnej dystrybucji danych (wszystkich funkcji i konfiguracji)
        sns.stripplot(
            data=df, x='Architektura', y='Czas_s', hue='T',
            palette=palette_colors, ax=axes[0], dodge=True, 
            jitter=0.15, size=2.5, edgecolor='black', linewidth=0.3, alpha=0.5
        )
        
        axes[0].set_title('a) Złożoność obliczeniowa', fontsize=9.5, fontweight='bold', loc='left')
        axes[0].set_xlabel('Klasa architektury sieciowej', fontsize=9)
        axes[0].set_ylabel('Czas pojedynczej inferencji [s]', fontsize=9)
        axes[0].grid(True, linestyle=':', alpha=0.5, axis='y')
        
        # Czyszczenie zduplikowanej legendy ze stripplota
        handles, labels = axes[0].get_legend_handles_labels()
        axes[0].legend(handles[:3], labels[:3], title='Kroki $T$', fontsize=7.5, title_fontsize=8, loc='upper left')
        
        # --- PANEL PRAWY: Jakość odszumiania (MSE) ---
        sns.boxplot(
            data=df, x='Architektura', y='MSE', hue='T', 
            palette=palette_colors, ax=axes[1], width=0.6, 
            linewidth=0.8, fliersize=0, boxprops=dict(alpha=0.85)
        )
        sns.stripplot(
            data=df, x='Architektura', y='MSE', hue='T',
            palette=palette_colors, ax=axes[1], dodge=True, 
            jitter=0.15, size=2.5, edgecolor='black', linewidth=0.3, alpha=0.5
        )
        
        # Skala logarytmiczna dla MSE jest kluczowa z uwagi na rzędy wielkości różnic
        axes[1].set_yscale('log')
        axes[1].set_title('b) Jakość odszumiania', fontsize=9.5, fontweight='bold', loc='left')
        axes[1].set_xlabel('Klasa architektury sieciowej', fontsize=9)
        axes[1].set_ylabel('Błąd średniokwadratowy MSE (skala log)', fontsize=9)
        axes[1].grid(True, linestyle=':', alpha=0.5, axis='y', which='both')
        
        # Czyszczenie zduplikowanej legendy
        handles2, labels2 = axes[1].get_legend_handles_labels()
        axes[1].legend(handles2[:3], labels2[:3], title='Kroki $T$', fontsize=7.5, title_fontsize=8, loc='upper right')
        
        # Usuwanie górnych i prawych krawędzi wykresów (styl czysty, "Paper-ready")
        for ax in axes:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(labelsize=8.5)
            
        plt.subplots_adjust(left=0.08, right=0.96, bottom=0.16, top=0.88, wspace=0.28)
        
        save_path = os.path.join(save_dir, 'magisterka_czytelne_podsumowanie_globalne.png')
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        print(f"\n[SUKCES] Wykres gotowy do publikacji zapisano w: {save_path}")

# Wywołanie funkcji
generate_clean_thesis_plots(test_functions, architectures_config)